In [1]:
!pip install --no-index --find-links=/kaggle/input/ariel-2024-pqdm pqdm

import pandas as pd
import numpy as np

from tqdm import tqdm
from pqdm.threads import pqdm
import itertools

from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

from astropy.stats import sigma_clip
from scipy.signal import savgol_filter

import matplotlib.pyplot as plt

from scipy import signal as scipy_signal
from scipy.stats import pearsonr

import time
__t0 = time.perf_counter()

class Config:
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"

    SCALE = 0.95
    SIGMA = 0.0009
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 7
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3

class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', index_col='planet_id').index.astype(int)

    def _apply_linear_corr(self, linear_corr, signal):
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]

        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]

        signal = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet").to_numpy()
        dark = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet").to_numpy()
        dead = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet").to_numpy()
        flat = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet").to_numpy()
        linear_corr = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet").values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])

        signal = signal.reshape(sensor_cfg["raw_shape"])
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset

        hot = sigma_clip(dark, sigma=5, maxiters=5).mask

        if sensor == "AIRS-CH0":
            signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]

        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]

        np.maximum(signal, 0, out=signal)

        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)

        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment

        signal[::2] -= dark * even_scale
        signal[1::2] -= dark * odd_scale
        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        mean_signal = np.nanmean(signal_roi, axis=1)

        cds_signal = mean_signal[1::2] - mean_signal[0::2]

        n_bins = cds_signal.shape[0] // binning
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])

        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)
            np.clip(binned, q_lo, q_hi, out=binned)

        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var

            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)

            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)

            binned *= w[None, :]
        return binned

    def _cross_correlate_signals(self, signal1, signal2, max_lag=None):
        """
        Compute cross-correlation between two 1D signals to find optimal alignment.
        
        Args:
            signal1, signal2: 1D arrays to cross-correlate
            max_lag: Maximum lag to consider (if None, uses min(len(signal1), len(signal2))//4)
        
        Returns:
            best_lag: Optimal lag for alignment
            max_corr: Maximum correlation coefficient
        """
        if max_lag is None:
            max_lag = min(len(signal1), len(signal2)) // 4
        
        # Handle edge cases
        if len(signal1) == 0 or len(signal2) == 0:
            return 0, 0.0
        
        # Normalize signals to zero mean for better correlation
        s1_norm = signal1 - np.nanmean(signal1)
        s2_norm = signal2 - np.nanmean(signal2)
        
        # Replace NaN values with 0 for correlation computation
        s1_norm = np.nan_to_num(s1_norm)
        s2_norm = np.nan_to_num(s2_norm)
        
        # Compute cross-correlation
        correlation = scipy_signal.correlate(s1_norm, s2_norm, mode='full')
        lags = scipy_signal.correlation_lags(len(s1_norm), len(s2_norm), mode='full')
        
        # Limit to specified max_lag
        valid_indices = np.abs(lags) <= max_lag
        correlation = correlation[valid_indices]
        lags = lags[valid_indices]
        
        # Find best lag
        best_idx = np.argmax(np.abs(correlation))
        best_lag = lags[best_idx]
        max_corr = correlation[best_idx]
        
        return best_lag, max_corr

    def _align_signals(self, signal1, signal2, lag):
        """
        Align two signals based on computed lag.
        
        Args:
            signal1, signal2: Signals to align
            lag: Lag computed from cross-correlation
        
        Returns:
            aligned_signal1, aligned_signal2: Aligned signals of same length
        """
        if lag == 0:
            min_len = min(signal1.shape[0], signal2.shape[0])
            return signal1[:min_len], signal2[:min_len]
        
        if lag > 0:
            # signal2 is ahead, trim beginning of signal2 and end of signal1
            aligned_signal1 = signal1[:-lag] if lag < len(signal1) else signal1[:0]
            aligned_signal2 = signal2[lag:]
        else:
            # signal1 is ahead, trim beginning of signal1 and end of signal2
            lag = abs(lag)
            aligned_signal1 = signal1[lag:]
            aligned_signal2 = signal2[:-lag] if lag < len(signal2) else signal2[:0]
        
        # Ensure same length
        min_len = min(aligned_signal1.shape[0], aligned_signal2.shape[0])
        return aligned_signal1[:min_len], aligned_signal2[:min_len]

    def _adaptive_weighted_fusion(self, airs_signal, fgs1_signal, window_size=50):
        """
        Perform adaptive weighted fusion based on local signal quality metrics.
        
        Args:
            airs_signal: AIRS-CH0 preprocessed signal
            fgs1_signal: FGS1 preprocessed signal  
            window_size: Window size for computing local weights
        
        Returns:
            fused_signal: Adaptively fused signal
        """
        if airs_signal.shape[0] == 0 or fgs1_signal.shape[0] == 0:
            return airs_signal if airs_signal.shape[0] > 0 else fgs1_signal
        
        n_samples = airs_signal.shape[0]
        fused_signal = np.zeros((n_samples, airs_signal.shape[1] + fgs1_signal.shape[1]))
        
        # Compute local signal quality metrics
        airs_snr = np.zeros(n_samples)
        fgs1_snr = np.zeros(n_samples)
        
        for i in range(n_samples):
            start_idx = max(0, i - window_size // 2)
            end_idx = min(n_samples, i + window_size // 2)
            
            # Compute signal-to-noise ratio as a quality metric
            if end_idx > start_idx:
                airs_window = airs_signal[start_idx:end_idx]
                fgs1_window = fgs1_signal[start_idx:end_idx]
                
                airs_mean = np.nanmean(airs_window)
                airs_std = np.nanstd(airs_window)
                airs_snr[i] = np.abs(airs_mean) / (airs_std + 1e-10)
                
                fgs1_mean = np.nanmean(fgs1_window)
                fgs1_std = np.nanstd(fgs1_window)
                fgs1_snr[i] = np.abs(fgs1_mean) / (fgs1_std + 1e-10)
        
        # Normalize SNR values to weights
        total_snr = airs_snr + fgs1_snr + 1e-10
        airs_weight = airs_snr / total_snr
        fgs1_weight = fgs1_snr / total_snr
        
        # Apply adaptive weighting and concatenate
        weighted_airs = airs_signal * airs_weight[:, np.newaxis]
        weighted_fgs1 = fgs1_signal * fgs1_weight[:, np.newaxis]
        
        fused_signal[:, :airs_signal.shape[1]] = weighted_airs
        fused_signal[:, airs_signal.shape[1]:] = weighted_fgs1
        
        return fused_signal

    def _cross_correlation_fusion(self, preprocessed_fgs1, preprocessed_airs_ch0):
        """
        Fuse AIRS-CH0 and FGS1 signals using cross-correlation based alignment.
        
        Args:
            preprocessed_fgs1: List of preprocessed FGS1 signals for each planet
            preprocessed_airs_ch0: List of preprocessed AIRS-CH0 signals for each planet
        
        Returns:
            fused_signals: Array of cross-correlation fused signals
        """
        fused_signals = []
        
        for i, (fgs1_signal, airs_signal) in enumerate(zip(preprocessed_fgs1, preprocessed_airs_ch0)):
            try:
                # Extract representative 1D signals for cross-correlation
                # For AIRS-CH0: use mean across spectral dimension
                if airs_signal.ndim > 1:
                    airs_1d = np.nanmean(airs_signal, axis=1)
                else:
                    airs_1d = airs_signal.copy()
                
                # For FGS1: already 1D after preprocessing
                if fgs1_signal.ndim > 1:
                    fgs1_1d = fgs1_signal.flatten() if fgs1_signal.shape[1] == 1 else np.nanmean(fgs1_signal, axis=1)
                else:
                    fgs1_1d = fgs1_signal.copy()
                
                # Compute cross-correlation to find optimal alignment
                lag, corr_coeff = self._cross_correlate_signals(airs_1d, fgs1_1d)
                
                # Align the full signals based on computed lag
                aligned_airs, aligned_fgs1 = self._align_signals(airs_signal, fgs1_signal, lag)
                
                # Perform adaptive weighted fusion
                if aligned_airs.shape[0] > 0 and aligned_fgs1.shape[0] > 0:
                    fused = self._adaptive_weighted_fusion(aligned_airs, aligned_fgs1)
                else:
                    # Fallback to concatenation if alignment fails
                    min_len = min(airs_signal.shape[0], fgs1_signal.shape[0])
                    airs_truncated = airs_signal[:min_len]
                    fgs1_truncated = fgs1_signal[:min_len]
                    fused = np.concatenate([fgs1_truncated, airs_truncated], axis=1)
                
                fused_signals.append(fused)
                
                # Optional: Log alignment quality for monitoring
                if hasattr(self.cfg, 'VERBOSE') and self.cfg.VERBOSE:
                    print(f"Planet {self.planet_ids[i]}: lag={lag}, corr={corr_coeff:.3f}")
                    
            except Exception as e:
                # Robust fallback to simple concatenation
                print(f"Warning: Cross-correlation fusion failed for planet {self.planet_ids[i] if i < len(self.planet_ids) else i}: {e}")
                min_len = min(airs_signal.shape[0], fgs1_signal.shape[0])
                airs_truncated = airs_signal[:min_len]
                fgs1_truncated = fgs1_signal[:min_len]
                fused = np.concatenate([fgs1_truncated, airs_truncated], axis=1)
                fused_signals.append(fused)
        
        return np.stack(fused_signals)

    def _process_planet_sensor(self, args):
        planet_id, sensor = args['planet_id'], args['sensor']
        calibrated = self._calibrate_single_signal(planet_id, sensor)
        preprocessed = self._preprocess_calibrated_signal(calibrated, sensor)
        return preprocessed

    def process_all_data(self):
        # Process both sensors in parallel as before
        args_fgs1 = [dict(planet_id=planet_id, sensor="FGS1") for planet_id in self.planet_ids]
        preprocessed_fgs1 = pqdm(args_fgs1, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        args_airs_ch0 = [dict(planet_id=planet_id, sensor="AIRS-CH0") for planet_id in self.planet_ids]
        preprocessed_airs_ch0 = pqdm(args_airs_ch0, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        # Apply cross-correlation based fusion instead of simple concatenation
        fused_signal = self._cross_correlation_fusion(preprocessed_fgs1, preprocessed_airs_ch0)
        
        return fused_signal
        
def _phase_detector_signal(signal, cfg):
    sl = cfg.MODEL_PHASE_DETECTION_SLICE
    min_idx = int(np.argmin(signal[sl])) + sl.start
    s1 = signal[:min_idx]; s2 = signal[min_idx:]
    if s1.size < 3 or s2.size < 3:
        return 0, len(signal) - 1
    g1 = np.gradient(s1); g1_max = np.max(g1) if np.size(g1) else 0.0
    g2 = np.gradient(s2); g2_max = np.max(g2) if np.size(g2) else 0.0
    if g1_max != 0: g1 /= g1_max
    if g2_max != 0: g2 /= g2_max
    phase1 = int(np.argmin(g1)); phase2 = int(np.argmax(g2)) + min_idx
    return phase1, phase2

def estimate_sigma_fgs(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    for single in preprocessed_data:
        air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
        p1, p2 = _phase_detector_signal(air_white, cfg)
        p1 = max(delta, p1)
        p2 = min(len(air_white) - delta - 1, p2)

        fgs = single[:, 0]
        oot = (fgs[: p1 - delta] if p1 - delta > 0 else np.empty(0, fgs.dtype))
        if p2 + delta < fgs.size:
            oot = np.concatenate([oot, fgs[p2 + delta :]])
        inn = fgs[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fgs))
        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.8, 1.25)

    return k * cfg.SIGMA

def estimate_sigma_air(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        white = np.nanmean(single[:, 1:], axis=1)
        white_s = savgol_filter(white, 20, 2)

        p1, p2 = _phase_detector_signal(white_s, cfg)
        p1 = max(delta, p1)
        p2 = min(len(white) - delta - 1, p2)

        oot_left = white[: p1 - delta] if p1 - delta > 0 else np.empty(0, white.dtype)
        oot_right = white[p2 + delta :] if (p2 + delta) < white.size else np.empty(0, white.dtype)
        oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
        inn = white[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.90, 1.20)

    return k * cfg.SIGMA

class TransitModel:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()
        
        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index

        return phase1, phase2
    
    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        
        return error

    def predict(self, single_preprocessed_signal):
        signal_1d = single_preprocessed_signal[:, 1:].mean(axis=1)
        signal_1d = savgol_filter(signal_1d, 20, 2)
        
        phase1, phase2 = self._phase_detector(signal_1d)

        phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, phase1)
        phase2 = min(len(signal_1d) - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, phase2)    

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d, phase1, phase2),
            method="Nelder-Mead"
        )
        
        return result.x[0]

    def predict_all(self, preprocessed_signals):
        predictions = [
            self.predict(preprocessed_signal)
            for preprocessed_signal in tqdm(preprocessed_signals)
        ]
        return np.array(predictions) * self.cfg.SCALE

class SubmissionGenerator:
    def __init__(self, config):
        self.cfg = config
        self.sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")

    def create(self, predictions, sigma_fgs=None, sigma_air=None):
        planet_ids = self.sample_submission.index
        n_mu = self.sample_submission.shape[1] // 2

        preds = np.asarray(predictions, dtype=float).reshape(-1)
        mu = np.tile(preds.reshape(-1, 1), (1, n_mu))
        mu = np.clip(mu, 0, None)

        sigmas = np.full_like(mu, self.cfg.SIGMA, dtype=float)
        if sigma_fgs is not None:
            sigma_fgs = np.asarray(sigma_fgs, dtype=float).reshape(-1)
            sigmas[:, 0] = np.clip(sigma_fgs, 1e-6, 0.1)
        if sigma_air is not None:
            sigma_air = np.asarray(sigma_air, dtype=float).reshape(-1, 1)
            sigmas[:, 1:] = np.clip(sigma_air, 1e-6, 0.1)

        submission_df = pd.DataFrame(
            np.concatenate([mu, sigmas], axis=1),
            columns=self.sample_submission.columns,
            index=planet_ids
        )
        submission_df.to_csv("submission.csv")
        return submission_df


config = Config()
    
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)

submission_generator = SubmissionGenerator(config)
submission = submission_generator.create(predictions, sigma_fgs=sigma_fgs_vec, sigma_air=sigma_air_vec)

__t1 = time.perf_counter()
elapsed = __t1 - __t0
print(f"[TIMING] total runtime: {elapsed:.2f} s ({elapsed/60:.2f} min)")

Looking in links: /kaggle/input/ariel-2024-pqdm
Processing /kaggle/input/ariel-2024-pqdm/pqdm-0.2.0-py2.py3-none-any.whl
Processing /kaggle/input/ariel-2024-pqdm/bounded_pool_executor-0.0.3-py3-none-any.whl (from pqdm)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 12.52it/s]

[TIMING] total runtime: 5.44 s (0.09 min)
